In [17]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
import MDAnalysis as mda

from mg_sampling import PROJECT_DIR
from mg_sampling.parallel import ParallelReader, ParallelAnalysis
from mg_sampling.analysis import IonNucleicAnalysis

In [19]:
TARGETS = ["W2385", "W2386"]
NREP = 12

In [23]:
for target in TARGETS:
    path = PROJECT_DIR / "casp17" / target
    topfile = path / "00" / "run.tpr"
    trajfiles = [
        path / f"{i:02d}" / "run.xtc"
        for i in range(NREP)
    ]
    indexfile = path / "00" / "bias_index.xvg"
    u = mda.Universe(topfile, trajfiles, format=ParallelReader)
    u.trajectory.demux(indexfile)  # This is in fact "remuxing"
    ana = IonNucleicAnalysis(u, "name MG", "nucleic", inner_cutoff=2.8, outer_cutoff=5).run()

  0%|          | 0/567 [00:00<?, ?it/s]

  0%|          | 0/758 [00:00<?, ?it/s]

In [26]:
df = ana.to_pandas()

In [27]:
import numpy as np
np.array({1:[1,2]}.keys())

array(dict_keys([1]), dtype=object)

In [43]:
def asunique(seq):
    """
    Ensure all elements of `seq` are equal and return the unique element.
    """
    it = iter(seq)
    x0 = next(it)
    for x in it:
        if x != x0:
            raise ValueError(f"Expected all elements to be equal ({x} != {x0})")
    return x0


def vectorize_dict_lookup(d: dict):
    """
    Return a function that accepts numpy arrays and maps
    them according to dictionary `d` of array-like values.
    All values of `d` must have the same shape.
    """
    dtype, shape = asunique((a.dtype, a.shape) for a in map(np.asarray, d.values()))
    dict_keys = np.array(list(d.keys()))
    dict_values = np.array(list(d.values()), dtype=dtype)

    sort_idx = np.argsort(dict_keys)
    sorted_keys = dict_keys[sort_idx]
    sorted_values = dict_values[sort_idx]

    def vectorized(keys):
        keys = np.asarray(keys)
        keys_shape = keys.shape
        flat_keys = keys.ravel()
        indices = np.searchsorted(sorted_keys, flat_keys, sorter=None)
        invalid = sorted_keys[np.clip(indices, 0, len(sorted_keys)-1)] != flat_keys
        if np.any(invalid):
            raise KeyError(flat_keys[invalid])
        flat_values = sorted_values[indices]
        values = flat_values.reshape((*keys_shape, *shape))
        return values
    
    return vectorized

In [48]:
d = {k: np.random.normal(size=(10, 10)) for k in "ABCDEFGHI"}

In [50]:
keys = np.array(list("ABCDEFGHI"))[np.random.choice(len("ABCDEFGHI"), size=(1000, 10))]

vectorize_dict_lookup(d)(keys)

array([[[[ 7.83789399e-01, -1.14470589e+00, -9.43015644e-01, ...,
          -3.47814427e-01,  5.30850792e-01,  6.94393088e-01],
         [-9.78013267e-01,  3.03809562e-01, -2.75731206e-01, ...,
           4.72389380e-01, -8.91785764e-01, -8.02730361e-01],
         [ 1.11705947e-01, -1.52514646e-01,  1.17246229e+00, ...,
          -2.12907286e-01, -7.76936615e-01, -1.95006470e+00],
         ...,
         [ 6.46995251e-01, -7.25854884e-01, -1.60075527e+00, ...,
          -8.26296173e-01, -3.58249288e-01, -6.23630803e-02],
         [ 1.03424325e+00,  1.49733594e+00,  4.54386668e-01, ...,
           2.90107239e-01,  2.11232640e-01, -1.93114661e-01],
         [ 2.61379278e-01, -7.73680012e-01, -1.43283975e-01, ...,
           1.42187201e+00, -7.00905187e-01, -1.76988783e-01]],

        [[-2.57738312e-01, -1.73809735e+00, -7.43979887e-01, ...,
          -2.27042782e-03,  1.10167527e+00,  1.66259658e+00],
         [-3.20915070e-01,  6.93179618e-01, -9.16824132e-01, ...,
           1.23959663e